**Direct Preference Optimization (DPO) | Industrial Maintenance Report Alignment**

Research question: Given our SFT-trained maintenance-report model, does preference optimization make it favor cautious, grounded reports over misleading ones?

Important: DPO is not reinforcement learning with a separately trained reward model. The paired chosen/`rejected` completions are used in a preference-based objective relative to a frozen snapshot of the SFT policy.

In [2]:
%pip install -q --upgrade unsloth datasets trl peft accelerate bitsandbytes pandas

**Step 1: Imports and GPU Check**

In [3]:
import unsloth
import json, re, random, gc, time, hashlib, inspect
from pathlib import Path
from collections import Counter
import torch
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported
from trl import DPOConfig, DPOTrainer
import transformers, trl, peft

assert torch.cuda.is_available(), 'Use Runtime → Change runtime type → T4 GPU.'
SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
print('GPU:', torch.cuda.get_device_name(0))
print('versions:', {'torch':torch.__version__, 'trl':trl.__version__, 'transformers':transformers.__version__, 'peft':peft.__version__})

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4
versions: {'torch': '2.11.0+cu130', 'trl': '1.15.0', 'transformers': '5.17.0', 'peft': '0.21.2'}


**Step 2: Drive mount and data verification**

In [4]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/llm-post-training-experimental-study')
DPO_FILE = PROJECT_ROOT / 'data/processed/dpo_prepared.jsonl'
SFT_ADAPTER = PROJECT_ROOT / 'models/stage2_domain_sft_adapter'  # Notebook 03 Branch B
DPO_ADAPTER = PROJECT_ROOT / 'models/stage3_dpo_adapter'
METRIC_DIR = PROJECT_ROOT / 'results/metrics'
COMPARISON_DIR = PROJECT_ROOT / 'results/comparisons'
LOCAL_LOGS = Path('/content/stage3_dpo_logs')
for folder in [METRIC_DIR, COMPARISON_DIR, LOCAL_LOGS]:folder.mkdir(parents=True, exist_ok=True)
assert DPO_FILE.is_file(), f'Run Notebook 01 and save {DPO_FILE}'
assert (SFT_ADAPTER / 'adapter_config.json').is_file(), f'Run Notebook 03 Branch B and save {SFT_ADAPTER}'
print('DPO dataset:', DPO_FILE)
print('Starting SFT adapter:', SFT_ADAPTER)

Mounted at /content/drive
DPO dataset: /content/drive/MyDrive/llm-post-training-experimental-study/data/processed/dpo_prepared.jsonl
Starting SFT adapter: /content/drive/MyDrive/llm-post-training-experimental-study/models/stage2_domain_sft_adapter


**Step 3: Configuration**

In [5]:
BASE_MODEL_NAME = 'unsloth/Qwen2.5-1.5B-bnb-4bit'
MAX_SEQ_LENGTH = 768
MAX_STEPS = 30
BATCH_SIZE = 1
GRAD_ACCUM = 4
LEARNING_RATE = 5e-6
BETA = 0.1
VAL_FRACTION = 0.15
assert MAX_STEPS > 0
print('Config:', {'max_steps':MAX_STEPS,'lr':LEARNING_RATE,'beta':BETA,'max_seq_length':MAX_SEQ_LENGTH})

Config: {'max_steps': 30, 'lr': 5e-06, 'beta': 0.1, 'max_seq_length': 768}


**Step 4: Load DPO data**

In [9]:
dpo_raw = load_dataset('json', data_files=str(DPO_FILE), split='train')
assert {'prompt','chosen','rejected'} <= set(dpo_raw.column_names), dpo_raw.column_names
assert len(dpo_raw) == 120, f'Expected our starter data to have 120 pairs; got {len(dpo_raw)}. Investigate changed source.'
assert all(all(isinstance(r[k],str) and r[k].strip() for k in ('prompt','chosen','rejected')) for r in dpo_raw)
assert all(r['chosen'] != r['rejected'] for r in dpo_raw)
assert all(r['prompt'].rstrip().endswith('### Response:') for r in dpo_raw)
print('Pairs:',len(dpo_raw),'| columns:',dpo_raw.column_names)
print('\nPROMPT:\n',dpo_raw[0]['prompt'][:600]);print('\nCHOSEN:\n',dpo_raw[0]['chosen'][:850]);print('\nREJECTED:\n',dpo_raw[0]['rejected'][:850])

Generating train split: 0 examples [00:00, ? examples/s]

Pairs: 120 | columns: ['prompt', 'chosen', 'rejected']

PROMPT:
 ### Instruction:
Prepare a structured maintenance handover from the note below. Separate evidence from hypotheses and make no unsupported claims.

Technician note: P-204: Vibration rose across two shifts; speed remained steady.

### Response:


CHOSEN:
 Equipment: P-204 (centrifugal pump)
Reported condition: Abnormal or uncertain equipment behavior noted; exact fault not confirmed.
Observed evidence: Vibration rose across two shifts; speed remained steady.
Possible contributors (unconfirmed): alignment, bearing condition, or structural looseness.
Missing information: verified vibration measurements, operating load, alignment history.
Recommended follow-up: Log vibration trend; request condition-monitoring review under approved procedures.
Assessment: Preliminary documentation only. Verify the observation and follow site-approved procedures before any diagnosis or intervention.

REJECTED:
 Equipment: P-204 (centrifugal pum

**Step 5: Data check**

In [10]:
for i in [0, 1, 15, 30, 60, 90, 119]:
    row=dpo_raw[i]
    print('\n'+'='*90+'\nPAIR',i)
    for k in ('prompt','chosen','rejected'):
        print(f'\n{k.upper()}:\n{row[k][:700]}')

identical = sum(r['chosen'].strip()==r['rejected'].strip() for r in dpo_raw)
confirmed_bad = sum('Confirmed root cause:' in r['rejected'] for r in dpo_raw)
print('\nExact identical pairs:', identical)
print('Rejected outputs with literal "Confirmed root cause:":',confirmed_bad)
print('Note: counts describe data patterns, NOT pairwise quality certification.')


PAIR 0

PROMPT:
### Instruction:
Prepare a structured maintenance handover from the note below. Separate evidence from hypotheses and make no unsupported claims.

Technician note: P-204: Vibration rose across two shifts; speed remained steady.

### Response:


CHOSEN:
Equipment: P-204 (centrifugal pump)
Reported condition: Abnormal or uncertain equipment behavior noted; exact fault not confirmed.
Observed evidence: Vibration rose across two shifts; speed remained steady.
Possible contributors (unconfirmed): alignment, bearing condition, or structural looseness.
Missing information: verified vibration measurements, operating load, alignment history.
Recommended follow-up: Log vibration trend; request condition-monitoring review under approved procedures.
Assessment: Preliminary documentation only. Verify the observation and follow site-approved procedures before any diagnosis or intervention.

REJECTED:
Equipment: P-204 (centrifugal pump)
Reported condition: Abnormal or uncertain equip

**Step 6: Split by ID to avoid duplicates**

In [11]:
def equipment_id(prompt):
    # For these generated prompts, the technical note includes identifiers like P-204.
    ids = re.findall(r'\b[A-Z]{1,6}-\d{2,5}\b', prompt)
    assert ids, f'Cannot extract equipment ID; inspect prompt: {prompt[:180]}'
    return ids[0]

ids=[equipment_id(p) for p in dpo_raw['prompt']]
counts=Counter(ids)
print('Unique equipment IDs:',len(counts),'| multiplicities:',Counter(counts.values()))
assert len(counts)==60, 'Expected 60 unique equipment groups; revisit extraction and grouping.'
groups=sorted(counts)
random.Random(SEED).shuffle(groups)
n_val=max(1,round(len(groups)*VAL_FRACTION))
val_ids=set(groups[:n_val]);train_ids=set(groups[n_val:])
assert not val_ids & train_ids
train_indices=[i for i,v in enumerate(ids) if v in train_ids]
val_indices=[i for i,v in enumerate(ids) if v in val_ids]
train_data=dpo_raw.select(train_indices).select_columns(['prompt','chosen','rejected'])
val_data=dpo_raw.select(val_indices).select_columns(['prompt','chosen','rejected'])
assert train_data and val_data
assert set(equipment_id(x) for x in train_data['prompt']).isdisjoint(set(equipment_id(x) for x in val_data['prompt']))
print('Train pairs:',len(train_data),'| validation pairs:',len(val_data))
print('Validation equipment IDs:',sorted(val_ids))

Unique equipment IDs: 60 | multiplicities: Counter({2: 60})
Train pairs: 102 | validation pairs: 18
Validation equipment IDs: ['F-018', 'FT-093', 'GB-145', 'HX-012', 'LT-260', 'M-514', 'WO-501', 'WO-622', 'WO-916']


**Step 7: Load Model**

In [12]:
from peft import PeftConfig
prior=PeftConfig.from_pretrained(str(SFT_ADAPTER))
print('Saved SFT adapter base:',prior.base_model_name_or_path)
assert 'qwen2.5' in str(prior.base_model_name_or_path).lower() and '1.5b' in str(prior.base_model_name_or_path).lower(), 'Wrong starting model: this is not the Qwen2.5-1.5B SFT adapter.'

model,tokenizer=FastLanguageModel.from_pretrained(
    model_name=str(SFT_ADAPTER), max_seq_length=MAX_SEQ_LENGTH,
    dtype=None, load_in_4bit=True,
)
if tokenizer.pad_token is None:tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='left'  # Required by recent TRL DPOTrainer
FastLanguageModel.for_inference(model)
print('Loaded adapters:', list(model.peft_config))
print('Padding:',tokenizer.padding_side,'| eos:',repr(tokenizer.eos_token))

Saved SFT adapter base: unsloth/Qwen2.5-1.5B-bnb-4bit
==((====))==  Unsloth 2026.11.2: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.11.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
Loaded adapters: ['default']
Padding: left | eos: '<|endoftext|>'


**Step 7: Test before training**

In [17]:
TEST_NOTE = (
    'Hydraulic press HP-417 had a pressure drop during three consecutive cycles. '
    'The cylinder returned slowly and a whining sound was noticed near the pump. '
    'A small oil stain was found under the unit; its source is unknown. '
    'No pressure readings, oil temperature, or maintenance history were available. '
    'The machine was stopped pending inspection.'
)
TASK = ('Transform the following technician note into a structured industrial maintenance report. '
        'Preserve all observed evidence, distinguish hypotheses from facts, identify unavailable information, '
        'and do not invent faults or measurements.\n\nTechnician note: ')
TEST_PROMPT = '### Instruction:\n' + TASK + TEST_NOTE + '\n\n### Response:\n'

def generate_report(prompt):
    FastLanguageModel.for_inference(model); model.eval()
    x=tokenizer(prompt,return_tensors='pt').to('cuda')
    with torch.inference_mode():
        y=model.generate(**x,max_new_tokens=260,do_sample=False,
                         pad_token_id=tokenizer.eos_token_id,
                         eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(y[0,x['input_ids'].shape[1]:],skip_special_tokens=True).strip()

sft_before = generate_report(TEST_PROMPT)
print('SFT — BEFORE DPO:\n', sft_before)
(COMPARISON_DIR/'stage3_hp417_before_dpo.json').write_text(
    json.dumps({'model':'SFT before DPO','prompt':TEST_PROMPT,'response':sft_before},indent=2),encoding='utf-8')

Both `max_new_tokens` (=260) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SFT — BEFORE DPO:
 Equipment: HP-417 (hydraulic press)
Reported issue: Pressure drop during hydraulic cycling.
Observed evidence: Pressure drop during three consecutive cycles; a small oil stain was found under the unit.
Possible contributors (not confirmed): leakage, flow restriction, or pressure sensor error.
Missing information: pressure readings, oil temperature, or maintenance history.
Recommended follow-up: Stop the equipment and arrange a hydraulic inspection.
Assessment: Preliminary documentation only; no definitive fault diagnosis is supported by this note.


1253

**Step 8: Add LORA adapter**

In [18]:
FastLanguageModel.for_training(model)
assert 'default' in model.peft_config, f'Unexpected adapter name(s): {list(model.peft_config)}. Current TRL expects "default" for reference duplication.'
model.set_adapter('default')
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Trainable parameters:',f'{trainable:,}')
assert trainable>0, 'All parameters are frozen; the SFT adapter must be trainable for DPO.'

def append_eos(r):
    return {key:r[key] if r[key].endswith(tokenizer.eos_token) else r[key]+tokenizer.eos_token for key in ('chosen','rejected')}
train_data=train_data.map(append_eos)
val_data=val_data.map(append_eos)

def pair_tokens(r):
    plen=len(tokenizer(r['prompt'],add_special_tokens=False)['input_ids'])
    chosen=plen+len(tokenizer(r['chosen'],add_special_tokens=False)['input_ids'])
    rejected=plen+len(tokenizer(r['rejected'],add_special_tokens=False)['input_ids'])
    return chosen,rejected
lengths=[pair_tokens(r) for r in list(train_data)+list(val_data)]
print('Max total prompt+response tokens:',max(max(t) for t in lengths))
print('Pairs exceeding max_length:',sum(max(t)>MAX_SEQ_LENGTH for t in lengths))
assert all(max(t)<=MAX_SEQ_LENGTH for t in lengths), 'Increase MAX_SEQ_LENGTH or inspect overlong pairs to prevent truncation.'

Trainable parameters: 18,464,768


Map:   0%|          | 0/102 [00:00<?, ? examples/s]

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

Max total prompt+response tokens: 185
Pairs exceeding max_length: 0


**Step 9: Configure DPO Trainer**

In [19]:
args = DPOConfig(
    output_dir=str(LOCAL_LOGS),
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    beta=BETA,
    loss_type='sigmoid',
    max_length=MAX_SEQ_LENGTH,
    logging_steps=1,
    warmup_steps=3,
    save_strategy='no',
    eval_strategy='no',
    report_to='none',
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim='adamw_8bit',
    seed=SEED,
    gradient_checkpointing=True,
)
trainer=DPOTrainer(
    model=model,
    ref_model=None,
    args=args,
    processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
)
print('PEFT adapter names:',list(model.peft_config))
assert 'ref' in model.peft_config, (
    'STOP: TRL did not create a reference adapter. Verify your TRL version and reference behavior '
    'before training, otherwise DPO may compare against raw base weights.')
ref_trainable=[name for name,p in model.named_parameters() if '.ref.' in name and p.requires_grad]
assert not ref_trainable, 'Reference adapter has trainable parameters; abort.'
policy_trainable=[name for name,p in model.named_parameters() if '.default.' in name and p.requires_grad]
assert policy_trainable, 'DPO policy is not trainable; abort.'
print('Frozen SFT reference verified. Trainable policy tensors:',len(policy_trainable))

Adding EOS to train dataset (num_proc=2):   0%|          | 0/102 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/102 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset (num_proc=2):   0%|          | 0/102 [00:00<?, ? examples…

Adding EOS to eval dataset (num_proc=2):   0%|          | 0/18 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=2):   0%|          | 0/18 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset (num_proc=2):   0%|          | 0/18 [00:00<?, ? examples/s…

PEFT adapter names: ['default', 'ref']
Frozen SFT reference verified. Trainable policy tensors: 392


**Step 10: Train DPO**

In [20]:
gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats();torch.cuda.synchronize()
t0=time.perf_counter()
train_result=trainer.train()
torch.cuda.synchronize()
train_seconds=time.perf_counter()-t0
peak_gib=torch.cuda.max_memory_allocated()/(1024**3)
print('Train loss:',train_result.training_loss)
print('Training seconds:',round(train_seconds,1))
print('Peak allocated GPU GiB:',round(peak_gib,2))

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 102 | Num Epochs = 2 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,580,643,840 (1.17% trained)


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logits / chosen,logits / rejected,logps / chosen,logps / rejected
1,0.693147,0.000000,0.000000,0.000000,0.000000,-0.262814,-0.223288,-237.142128,-270.039482
2,0.693147,0.000000,0.000000,0.000000,0.000000,-0.288540,-0.327022,-210.072899,-206.857201
3,0.670408,0.025442,-0.020917,1.000000,0.046359,-0.312349,-0.327487,-228.496998,-244.227489
4,0.661991,0.084378,0.020421,1.000000,0.063957,-0.328954,-0.247631,-207.353336,-243.177914
5,0.565162,0.144820,-0.132328,1.000000,0.277148,-0.371971,-0.337613,-231.558037,-267.438190
6,0.550320,0.205288,-0.118437,1.000000,0.323725,-0.374006,-0.368262,-232.944073,-255.006763
7,0.543700,0.204547,-0.130977,1.000000,0.335524,-0.424185,-0.466471,-220.394825,-217.809628
8,0.293004,0.293995,-0.787073,1.000000,1.081068,-0.382970,-0.427411,-224.452168,-257.427490
9,0.426514,0.422062,-0.238556,1.000000,0.660618,-0.449510,-0.472500,-219.503983,-228.640213
10,0.442644,0.364607,-0.299618,1.000000,0.664225,-0.396478,-0.363256,-218.565613,-247.206627


Train loss: 0.4157868484656016
Training seconds: 109.1
Peak allocated GPU GiB: 1.45


**Evaluation**

In [21]:
eval_metrics=trainer.evaluate()
print(json.dumps({k:float(v) if isinstance(v,(int,float)) else str(v) for k,v in eval_metrics.items()},indent=2))

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.209728,0.259862,30,1.344480,37582.000000,-0.525659,-0.473991,0.626195,0.440502,-1.000919,1.000000,1.441421,-228.921610,-273.098512


{
  "eval_loss": 0.25986191630363464,
  "eval_entropy": 1.3444797926478915,
  "eval_num_tokens": 37582.0,
  "eval_logits/chosen": -0.5256587083547418,
  "eval_logits/rejected": -0.47399106869890734,
  "eval_mean_token_accuracy": 0.6261948380205367,
  "eval_rewards/chosen": 0.4405016605224874,
  "eval_rewards/rejected": -1.0009191933398445,
  "eval_rewards/accuracies": 1.0,
  "eval_rewards/margins": 1.44142085313797,
  "eval_logps/chosen": -228.92160966661243,
  "eval_logps/rejected": -273.0985124376085
}


**Save DPO adaptor**

In [22]:
DPO_ADAPTER.mkdir(parents=True,exist_ok=True)
# Save just the DPO-optimized policy; not the reference adapter.
model.save_pretrained(str(DPO_ADAPTER), selected_adapters=['default'])
tokenizer.save_pretrained(str(DPO_ADAPTER))
assert (DPO_ADAPTER/'adapter_config.json').is_file(), 'Saved policy adapter missing config.'
run_metrics={
    'model':'Qwen2.5-1.5B','starting_adapter':str(SFT_ADAPTER),
    'saved_dpo_adapter':str(DPO_ADAPTER),'dpo_dataset':str(DPO_FILE),
    'train_pairs':len(train_data),'validation_pairs':len(val_data),
    'unique_train_equipment_ids':len(train_ids),'unique_validation_equipment_ids':len(val_ids),
    'seed':SEED,'beta':BETA,'learning_rate':LEARNING_RATE,'max_steps':MAX_STEPS,
    'batch_size':BATCH_SIZE,'gradient_accumulation_steps':GRAD_ACCUM,
    'max_length':MAX_SEQ_LENGTH,'train_loss':float(train_result.training_loss),
    'eval_metrics':{k:float(v) if isinstance(v,(int,float)) else str(v) for k,v in eval_metrics.items()},
    'training_seconds':train_seconds,'peak_gpu_memory_allocated_gib':peak_gib,
    'gpu':torch.cuda.get_device_name(0),
    'interpretation_warning':'Small synthetic preference corpus; preference metrics are not factuality or observation recall.'
}
METRIC_FILE=METRIC_DIR/'stage3_dpo_metrics.json'
METRIC_FILE.write_text(json.dumps(run_metrics,indent=2),encoding='utf-8')
print('Saved DPO adapter:',DPO_ADAPTER)
print('Saved experiment metrics:',METRIC_FILE)

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/llm-post-training-experimental-study/models/stage3_dpo_adapter/tokenizer_config.json.


Saved DPO adapter: /content/drive/MyDrive/llm-post-training-experimental-study/models/stage3_dpo_adapter
Saved experiment metrics: /content/drive/MyDrive/llm-post-training-experimental-study/results/metrics/stage3_dpo_metrics.json


**Inference**

In [ ]:
model.set_adapter('default')
dpo_after=generate_report(TEST_PROMPT)
print('SFT BEFORE DPO:\n',sft_before)
print('\n'+'='*90+'\nDPO AFTER:\n',dpo_after)
comparison={'prompt':TEST_PROMPT,'sft_before':sft_before,'dpo_after':dpo_after,
            'warning':'Preliminary side-by-side example; no causal superiority claim from one output.'}
comparison_file=COMPARISON_DIR/'stage3_hp417_sft_vs_dpo.json'
comparison_file.write_text(json.dumps(comparison,indent=2),encoding='utf-8')
print('\nComparison saved:',comparison_file)

Both `max_new_tokens` (=260) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SFT BEFORE DPO:
 Equipment: HP-417 (hydraulic press)
Reported issue: Pressure drop during hydraulic cycling.
Observed evidence: Pressure drop during three consecutive cycles; a small oil stain was found under the unit.
Possible contributors (not confirmed): leakage, flow restriction, or pressure sensor error.
Missing information: pressure readings, oil temperature, or maintenance history.
Recommended follow-up: Stop the equipment and arrange a hydraulic inspection.
Assessment: Preliminary documentation only; no definitive fault diagnosis is supported by this note.

DPO AFTER:
 Equipment: P-829 (centrifugal pump)
Reported issue: Reduced discharge flow.
Observed evidence: intermittent flow reduction and vibration near the pump casing.
Possible contributors (not confirmed): cavitation, suction conditions, or mechanical issues.
Missing information: flow meter readings, suction conditions, vibration trend, and verified pressure and flow measurements.
Recommended follow-up: Request a qualifi

In [ ]:
print(trainer.state.global_step)
print(trainer.state.log_history[-5:])

30
[{'loss': 0.21627967059612274, 'grad_norm': 6.176536560058594, 'learning_rate': 5.555555555555555e-07, 'entropy': 1.3399736285209656, 'num_tokens': 35000.0, 'logits/chosen': -0.6041400277957233, 'logits/rejected': -0.5465742214991516, 'mean_token_accuracy': 0.642227366566658, 'rewards/chosen': 0.513636402785778, 'rewards/rejected': -1.0037247017025948, 'rewards/accuracies': 1.0, 'rewards/margins': 1.5173611342906952, 'logps/chosen': -203.50030899047852, 'logps/rejected': -248.30878448486328, 'epoch': 1.0784313725490196, 'step': 28}, {'loss': 0.24073684215545654, 'grad_norm': 5.806943893432617, 'learning_rate': 3.7037037037037036e-07, 'entropy': 1.371499925851822, 'num_tokens': 36278.0, 'logits/chosen': -0.4853750707884932, 'logits/rejected': -0.45049515031707, 'mean_token_accuracy': 0.6255502849817276, 'rewards/chosen': 0.3837593160569668, 'rewards/rejected': -1.0106838420033455, 'rewards/accuracies': 1.0, 'rewards/margins': 1.394443154335022, 'logps/chosen': -228.83746337890625, 'l

In [15]:
test_prompt = """### Instruction:
Transform the following technician note into a structured industrial maintenance report. Clearly distinguish observed evidence from possible causes, identify missing information, and do not invent measurements or confirm unverified faults.

Technician note:
During the afternoon shift, centrifugal pump P-829 experienced intermittent reductions in discharge flow over approximately 40 minutes. The operator noticed increased vibration near the pump casing and a faint rattling sound during operation. The discharge pressure gauge displayed 3.8 bar, which was within the normal operating range, but no flow meter readings were recorded. The operator suspected cavitation, although this was not verified. A maintenance inspection conducted two days earlier reported no visible external leakage. The pump remained operational under monitoring pending further assessment.

### Response:
"""

In [23]:
def generate_report(model, tokenizer, prompt):
    FastLanguageModel.for_inference(model)
    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(next(model.parameters()).device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

In [26]:
from unsloth import FastLanguageModel
from pathlib import Path

DPO_ADAPTER = Path(
    "/content/drive/MyDrive/llm-post-training-experimental-study/models/stage3_dpo_adapter"
)

assert (DPO_ADAPTER / "adapter_config.json").exists(), (
    f"DPO adapter not found at {DPO_ADAPTER}. "
    "Check the actual save path in Notebook 04."
)

dpo_model, dpo_tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(DPO_ADAPTER),
    max_seq_length=768,
    dtype=None,
    load_in_4bit=True
)

FastLanguageModel.for_inference(dpo_model)
dpo_model.eval()

print("DPO model loaded successfully!")

==((====))==  Unsloth 2026.11.2: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
DPO model loaded successfully!


In [28]:
sft_output = generate_report(
    model,
    tokenizer,
    test_prompt
)

dpo_output = generate_report(
    dpo_model,
    tokenizer,
    test_prompt
)

print("SFT BEFORE DPO:\n", sft_output)
print("\n" + "=" * 80)
print("DPO AFTER:\n", dpo_output)

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SFT BEFORE DPO:
 Equipment: P-829 (centrifugal pump)
Reported issue: Reduced discharge flow.
Observed evidence: intermittent flow reduction and vibration near the pump casing.
Possible contributors (not confirmed): cavitation, suction conditions, or mechanical issues.
Missing information: flow meter readings, suction conditions, vibration trend, and verified pressure and flow measurements.
Recommended follow-up: Request a qualified pump inspection and suction condition review.
Assessment: Preliminary documentation only; no definitive fault diagnosis is supported by this note.

DPO AFTER:
 Equipment: P-829 (centrifugal pump)
Reported issue: Reduced discharge flow.
Observed evidence: intermittent flow reduction and vibration near the pump casing.
Possible contributors (not confirmed): cavitation, suction conditions, or mechanical issues.
Missing information: flow meter readings, suction conditions, vibration trend, and verified pressure and flow measurements.
Recommended follow-up: Reque

Right now, we see the SFT and DPO models both give the same result. I am going to run some code to see some checkpoints.

In [29]:
from safetensors import safe_open
from pathlib import Path
import torch

root = Path(
    "/content/drive/MyDrive/llm-post-training-experimental-study/models"
)

sft_path = root / "stage2_domain_sft_adapter"
dpo_path = root / "stage3_dpo_adapter"

def load_adapter_weights(folder):
    files = sorted(folder.glob("*.safetensors"))
    assert files, f"No adapter weights found in {folder}"

    weights = {}
    for file in files:
        with safe_open(str(file), framework="pt", device="cpu") as f:
            for key in f.keys():
                weights[key] = f.get_tensor(key)
    return weights

sft_weights = load_adapter_weights(sft_path)
dpo_weights = load_adapter_weights(dpo_path)

common = set(sft_weights) & set(dpo_weights)

assert common, "No matching weight keys between SFT and DPO!"

changed = 0
max_difference = 0.0

for key in common:
    a = sft_weights[key].float()
    b = dpo_weights[key].float()

    assert a.shape == b.shape, f"Shape mismatch: {key}"

    difference = (a - b).abs().max().item()

    if difference > 0:
        changed += 1

    max_difference = max(max_difference, difference)

print("Matching tensors:", len(common))
print("Changed tensors:", changed)
print("Maximum absolute weight difference:", max_difference)

if changed > 0:
    print("PASS: The saved DPO adapter differs from SFT.")
else:
    print("WARNING: No weight changes detected.")

Matching tensors: 392
Changed tensors: 392
Maximum absolute weight difference: 0.0015405267477035522
PASS: The saved DPO adapter differs from SFT.


All 392 matching tensors have changed between the saved SFT and DPO adapters.
We now have strong evidence that DPO optimization occurred and updated the adapter weights.

In [30]:
TEST_PROMPT2 = """### Instruction:
Transform the following technician note into a structured industrial maintenance report. Clearly distinguish observations from suspected causes. Do not invent measurements or confirm unverified faults.

Technician note:
Air compressor AC-936 shut down unexpectedly during the night shift.
The operator heard a loud metallic knocking sound shortly before shutdown.
The operator insists that the crankshaft has definitely failed and must be
replaced immediately. However, no internal inspection has been performed.
No vibration measurements, oil analysis, temperature readings, or diagnostic
tests are available. The compressor remains isolated pending inspection.

### Response:
"""

In [31]:
sft_output = generate_report(
    model,
    tokenizer,
    TEST_PROMPT2)

dpo_output = generate_report(
    dpo_model,
    tokenizer,
    TEST_PROMPT2
)

print("SFT BEFORE DPO:\n", sft_output)
print("\n" + "=" * 80)
print("DPO AFTER:\n", dpo_output)

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SFT BEFORE DPO:
 Equipment: AC-936 (air compressor)
Reported issue: Unplanned shutdown.
Observed evidence: The operator heard a loud metallic knocking sound shortly before shutdown.
Possible contributors (not confirmed): mechanical damage, lubrication, or pressure transients.
Missing information: vibration, oil analysis, temperature readings, or diagnostic tests.
Recommended follow-up: Request authorized mechanical inspection and authorized pressure monitoring.
Assessment: Preliminary documentation only; no definitive fault diagnosis is supported by this note.

DPO AFTER:
 Equipment: AC-936 (air compressor)
Reported issue: Unplanned shutdown.
Observed evidence: The operator heard a loud metallic knocking sound shortly before shutdown.
Possible contributors (not confirmed): mechanical damage, lubrication, or pressure transients.
Missing information: vibration, oil analysis, temperature readings, or diagnostic tests.
Recommended follow-up: Request authorized mechanical inspection and aut

### Final Observation — DPO vs SFT

DPO training completed successfully, with a validation reward accuracy of 100% and a positive reward margin of approximately 1.44 on the synthetic preference dataset.

A direct comparison of the saved SFT and DPO adapters confirmed that all 392 corresponding parameter tensors had changed, with a maximum absolute weight difference of approximately 0.00154.

However, qualitative inference tests using deterministic decoding did not reveal observable differences in generated maintenance reports.

These findings suggest that preference optimization changed the model parameters and improved its discrimination of preferred versus rejected responses within the training objective, without demonstrating a corresponding improvement in generated report quality.

Further evaluation using held-out preference pairs and systematic report-quality metrics is necessary to determine whether the learned preferences generalize beyond the synthetic training examples.

The identical generated outputs are retained as an experimental observation rather than treated as evidence of implementation failure.